# Preflight de hardware y persistencia para el piloto (sin entrenamiento)

Esta auditoría comprueba la GPU realmente asignada, memoria CUDA, RAM, almacenamiento temporal y escritura/lectura de un checkpoint sintético en el Google Drive montado. No descarga datasets, no accede a test, no ejecuta modelos ni realiza `backward()`/`optimizer.step()`. El resultado de Drive solo prueba I/O en el montaje actual; no garantiza persistencia después de destruir la VM.

**No activa `RUN_TRAINING`.** Un resultado CPU-only permite revisar Drive, pero no aprueba el preflight GPU.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil

drive.mount('/content/drive')
OUT_DRIVE = Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_entorno_entrenamiento')
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive no está montado correctamente.')
print('Destino de informes:', OUT_DRIVE)


## 2. Actualizar el repositorio privado

Se utiliza un PAT con permiso **Contents: Read-only**. Puedes guardarlo en Colab Secrets como `GITHUB_TOKEN`; si no, se solicitará de forma oculta. No pegues el token en una celda ni lo compartas en el chat.

In [ ]:
from getpass import getpass
REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass('GitHub PAT (Contents: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta el token GitHub.')
askpass=Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\\ncase "$1" in\\n *Username*) echo x-access-token ;;\\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\\n *) exit 1 ;;\\nesac\\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN, env
os.chdir(WORKDIR)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:', HEAD)
if not Path('scripts/audit_training_runtime.py').is_file():
    raise RuntimeError('No existe el auditor de entorno en este HEAD.')


## 3. Diagnóstico de GPU, RAM, disco y checkpoint sintético

El script realiza una reserva temporal pequeña de memoria GPU (64 MiB por defecto) y la libera. En Drive escribe un checkpoint sintético pequeño, lo renombra, lo vuelve a leer, compara tensor y SHA-256 y elimina el archivo de prueba. No contiene pesos ni datos del proyecto.

In [ ]:
cmd=[sys.executable,'scripts/audit_training_runtime.py','--output-dir',str(OUT_DRIVE),'--gpu-probe-mib','64']
proc=subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)
print(proc.stdout)
if proc.stderr: print('STDERR:',proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'Falló la auditoría de entorno (exit={proc.returncode}). Revisa el JSON/stdout antes de continuar.')
report_path=OUT_DRIVE/'training_runtime_preflight.json'
if not report_path.is_file(): raise RuntimeError('No se generó el informe en Drive.')
report=json.loads(report_path.read_text(encoding='utf-8'))
print('Informe persistido:', report_path)


## 4. Dictamen automático

- `DRIVE_IO_PASS` confirma escritura/lectura en el montaje actual, no recuperación tras reiniciar la VM.
- `GPU_PASS` exige CUDA y que la reserva de memoria sintética funcione.
- `PREFLIGHT_READY` exige ambos. Incluso si aparece, solo habilita la siguiente revisión de memoria por modelo/batch; **no autoriza entrenamiento**.

In [ ]:
drive_ok=report.get('drive_checkpoint_io',{}).get('status')=='PASS'
gpu_ok=report.get('cuda_available') is True and report.get('gpu_allocation_probe',{}).get('status')=='PASS'
print('DRIVE_IO_PASS:', drive_ok)
print('GPU_PASS:', gpu_ok)
print('PREFLIGHT_READY:', drive_ok and gpu_ok)
print('TRAINING_STARTED:', report.get('training_started'))
print('TEST_SPLIT_TOUCHED:', report.get('test_split_touched'))
print('Si GPU_PASS es False, no ejecutes el piloto: cambia el acelerador de Colab a GPU y vuelve a ejecutar desde la primera celda.')
